# 05 — Resource Benchmarking

Measure resources from the actual training and inference runs. Record hardware/software metadata and keep one row per experiment. The notebook can aggregate multiple SLMs when adapters and model checkpoints are available.

In [ ]:
import os, time, json, platform, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import psutil

## Hardware/software fingerprint

In [ ]:
def environment_report():
    report = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "cpu": platform.processor(),
        "ram_total_gb": round(psutil.virtual_memory().total / 1024**3, 2),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
    }
    if torch.cuda.is_available():
        report["cuda_version"] = torch.version.cuda
        report["gpu_name"] = torch.cuda.get_device_name(0)
        report["gpu_count"] = torch.cuda.device_count()
    return report

environment = environment_report()
print(json.dumps(environment, indent=2))

## Memory measurement helpers

In [ ]:
def reset_gpu_peak():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def gpu_memory_snapshot():
    if not torch.cuda.is_available():
        return {"gpu_available": False}
    return {
        "gpu_available": True,
        "allocated_gb": torch.cuda.memory_allocated() / 1024**3,
        "reserved_gb": torch.cuda.memory_reserved() / 1024**3,
        "peak_allocated_gb": torch.cuda.max_memory_allocated() / 1024**3,
        "peak_reserved_gb": torch.cuda.max_memory_reserved() / 1024**3,
    }

## Training-time benchmark schema

In [ ]:
BENCHMARK_COLUMNS = [
    "run_id", "model_id", "method", "dataset_version",
    "training_seconds", "tokens_per_second", "samples_per_second",
    "peak_allocated_gb", "peak_reserved_gb", "cpu_ram_peak_gb",
    "trainable_parameters", "total_parameters", "trainable_percent",
    "adapter_size_mb", "gpu_name", "timestamp_utc",
]

## Parameter count

In [ ]:
def parameter_stats(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "total_parameters": total,
        "trainable_parameters": trainable,
        "trainable_percent": 100 * trainable / total if total else 0,
    }

## Adapter disk size

In [ ]:
def directory_size_mb(directory):
    directory = Path(directory)
    if not directory.exists():
        return None
    return sum(p.stat().st_size for p in directory.rglob("*") if p.is_file()) / (1024**2)

## Inference latency benchmark

In [ ]:
@torch.inference_mode()
def benchmark_callable(fn, warmup=3, repetitions=20):
    for _ in range(warmup):
        fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    durations = []
    for _ in range(repetitions):
        start = time.perf_counter()
        fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        durations.append(time.perf_counter() - start)
    return {
        "mean_seconds": float(np.mean(durations)),
        "median_seconds": float(np.median(durations)),
        "p95_seconds": float(np.percentile(durations, 95)),
        "std_seconds": float(np.std(durations)),
        "repetitions": repetitions,
    }

## CSV/JSON run registry

In [ ]:
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)
REGISTRY_PATH = RESULTS_DIR / "benchmark_runs.csv"

def append_benchmark(record):
    row = pd.DataFrame([record])
    row.to_csv(REGISTRY_PATH, mode="a", header=not REGISTRY_PATH.exists(), index=False)
    return row

# Capture a real benchmark record after each completed experiment.

## Load benchmark history

In [ ]:
def load_benchmark_history(path=REGISTRY_PATH):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(columns=BENCHMARK_COLUMNS)
    return pd.read_csv(path)

# history = load_benchmark_history()
# history.tail()

## Visualize resource trade-offs

In [ ]:
def plot_metric_comparison(frame, metric, title=None):
    if frame.empty or metric not in frame.columns:
        raise ValueError("Provide a populated benchmark dataframe and a valid metric.")
    summary = frame.groupby(["model_id", "method"], as_index=False)[metric].mean()
    summary.plot(kind="bar", x="method", y=metric, legend=False, figsize=(8, 4))
    plt.title(title or metric)
    plt.ylabel(metric)
    plt.tight_layout()
    plt.show()

# plot_metric_comparison(history, "peak_allocated_gb", "Peak GPU memory")

## Export reproducibility package

In [ ]:
def save_run_manifest(config, destination):
    manifest = {
        "environment": environment_report(),
        "config": config,
        "notes": "Populate dataset/model hashes and git commit before publishing.",
    }
    with open(destination, "w") as fp:
        json.dump(manifest, fp, indent=2)

# save_run_manifest(CONFIG, RESULTS_DIR / "run_manifest.json")

## Verified environment report

Actual runtime details captured while preparing this notebook. GPU-specific fields are absent because CUDA is unavailable in this runtime.

In [1]:
print(json.dumps(environment_report(), indent=2))


{
  "python": "3.13.5",
  "platform": "Linux-6.18.57-x86_64-with-glibc2.41",
  "cpu": "",
  "ram_total_gb": 5.81,
  "torch": "2.10.0+cpu",
  "cuda_available": false
}

## Benchmark helper self-test (CPU only)

The following measures a tiny local tensor operation to verify the timing helper. These timings are **not** LLM inference latency and must not be compared with model results.

In [2]:
def tiny_cpu_workload():
    values = torch.arange(1000, dtype=torch.float32)
    return (values * values).sum().item()

latency = benchmark_callable(tiny_cpu_workload, warmup=3, repetitions=20)
print(json.dumps(latency, indent=2))


{
  "mean_seconds": 6.889999986015027e-06,
  "median_seconds": 6.425999913517444e-06,
  "p95_seconds": 8.068549936979257e-06,
  "std_seconds": 1.612220275249062e-06,
  "repetitions": 20
}

## Parameter-count helper self-test

A small toy PyTorch network is used to verify counting logic. These are not parameter counts for the healthcare SLM.

In [3]:
toy_model = torch.nn.Sequential(
    torch.nn.Linear(4, 3),
    torch.nn.ReLU(),
    torch.nn.Linear(3, 2)
)
print(parameter_stats(toy_model))


{
  "total_parameters": 23,
  "trainable_parameters": 23,
  "trainable_percent": 100.0
}

## Adapter-size helper self-test

A temporary binary fixture checks the directory-size function. This is not a trained adapter size.

In [4]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as temp_dir:
    test_file = Path(temp_dir) / "adapter-test.bin"
    test_file.write_bytes(b"benchmark-fixture" * 128)
    print(f"Temporary fixture size: {directory_size_mb(temp_dir):.6f} MB")


Temporary fixture size: 0.002075 MB

## Empty benchmark registry check

If no experiments have been recorded, the loader returns an empty table with the expected schema rather than inventing run rows.

In [5]:
empty_history = pd.DataFrame(columns=BENCHMARK_COLUMNS)
print("Rows:", len(empty_history))
print("Columns:", list(empty_history.columns))


Rows: 0
Columns: ['run_id', 'model_id', 'method', 'dataset_version', 'training_seconds', 'tokens_per_second', 'samples_per_second', 'peak_allocated_gb', 'peak_reserved_gb', 'cpu_ram_peak_gb', 'trainable_parameters', 'total_parameters', 'trainable_percent', 'adapter_size_mb', 'gpu_name', 'timestamp_utc']

## Benchmark interpretation

The CPU helper checks validate timing, parameter-count, file-size, and empty-registry utilities only. They do not constitute LoRA/QLoRA training benchmarks. Real model metrics must be appended from completed training and inference runs.